1. Fetch NYT API          → valider les credentials, le format de réponse
2. Nettoyage / features   → ce qui entre dans FinBERT
3. Inférence FinBERT      → scores de sentiment, distributions
4. Agrégation temporelle  → features pour Prophet
5. Train Prophet          → forecast sur 7 jours
6. Log MLflow             → un run complet, les deux modèles enregistrés

In [1]:
from pipeline.core.src.sql import init_table_articles

---
## Modeling

In [ ]:
from transformers import pipeline
pipe = pipeline("text-classification", model="ProsusAI/finbert")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 19453.02it/s]


In [ ]:
results = [pipe(text, truncation=True, max_length=512)[0] for text in df_business['text']]
df_business['sentiment_label'] = [r['label'] for r in results]
df_business['sentiment_score'] = [r['score'] for r in results]
df_business

,date,text,sentiment_label,sentiment_score
0,2024-01-01,The tentative deal for the men’s golf circuits...,neutral,0.855811
1,2024-01-01,The slowdown in the residential real estate ma...,negative,0.969375
2,2024-01-01,"Sales by BYD, the country’s dominant automaker...",positive,0.917731
3,2024-01-02,Long-term investment in India by businesses is...,positive,0.932099
4,2024-01-02,The power grid can’t keep up with demand for c...,neutral,0.361800
...,...,...,...,...
265,2024-01-31,"The retail giant, which last opened a domestic...",positive,0.841143
266,2024-01-31,The manufacturer is under pressure to improve ...,negative,0.959776
267,2024-01-31,The company has been renegotiating the contrac...,negative,0.931435
268,2024-01-31,"The move, which involves the Fenway Sports Gro...",positive,0.902623


1m35 pour 270 rows, pour un mois.

270 articles → 95 secondes → ~0.35s par article.

on peut imaginer que c'est la moyenne mensuelle, donc on a un temps de traitement de 19 minutes pour le volume annuel (270*12=3240)

57 minutes sur 3 ans